# Day 7 — Finalization, Git Workflow & ML Presentation





## 1. Project setup
**What:** identify the project root and standard folders.

**Next:** load available project outputs.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
OUTPUTS = ROOT / "outputs"
MODELS = ROOT / "models"
FINAL = OUTPUTS / "day7_finalization"
FINAL.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
for folder in [RAW, PROCESSED, OUTPUTS, MODELS]:
    print(f"{folder.relative_to(ROOT)}: {'Found' if folder.exists() else 'Missing'}")


Project root: /home/nineleaps/Downloads/uber_eats-ML
data/raw: Found
data/processed: Found
outputs: Found
models: Found


## 2. Check required raw datasets
**What:** check whether the expected synthetic CSV files exist and inspect their dimensions.

**Result:** missing files are listed for correction.

**Next:** verify analytical outputs.

In [2]:
required_raw = ["customers.csv", "restaurants.csv", "orders.csv",
                "deliveries.csv", "reviews.csv", "payments.csv", "drivers.csv"]
raw_checks = []
for filename in required_raw:
    path = RAW / filename
    if path.exists():
        try:
            frame = pd.read_csv(path)
            raw_checks.append({"file": filename, "status": "Available",
                               "rows": len(frame), "columns": len(frame.columns)})
        except Exception as exc:
            raw_checks.append({"file": filename, "status": f"Unreadable: {exc}",
                               "rows": None, "columns": None})
    else:
        raw_checks.append({"file": filename, "status": "Missing",
                           "rows": None, "columns": None})
raw_status = pd.DataFrame(raw_checks)
display(raw_status)
raw_status.to_csv(FINAL / "raw_dataset_check.csv", index=False)


,file,status,rows,columns
0,customers.csv,Available,300,5
1,restaurants.csv,Available,45,6
2,orders.csv,Available,1800,8
3,deliveries.csv,Available,1800,9
4,reviews.csv,Available,1147,6
5,payments.csv,Available,1800,5
6,drivers.csv,Available,70,4


## 3. Check deliverables by project day
**What:** look for the main notebook and evidence files created in earlier days.

**Result:** this is a file-presence check, not a quality certification.

**Next:** inspect model metrics where available.

In [3]:
expected_files = [
    ("Day 2 customer segmentation notebook", ROOT/"notebooks/customer_segmentation.ipynb"),
    ("Day 2 restaurant segmentation notebook", ROOT/"notebooks/restaurant_segmentation.ipynb"),
    ("Day 3 NLP notebook", ROOT/"notebooks/day3_customer_feedback_nlp.ipynb"),
    ("Day 4 delivery prediction notebook", ROOT/"notebooks/day4_delivery_time_prediction.ipynb"),
    ("Day 5 demand forecasting notebook", ROOT/"notebooks/day5_hourly_demand_forecasting.ipynb"),
    ("Day 6 business insights notebook", ROOT/"notebooks/day6_end_to_end_ml_pipeline_business_insights.ipynb"),
    ("Data dictionary", ROOT/"docs/data_dictionary.md"),
    ("Project README", ROOT/"README.md"),
    ("Python requirements", ROOT/"requirements.txt"),
    ("Git ignore rules", ROOT/".gitignore"),
]
deliverable_check = pd.DataFrame([
    {"deliverable": name, "status": "Available" if path.exists() else "Not found",
     "path": str(path.relative_to(ROOT))}
    for name, path in expected_files
])
display(deliverable_check)
deliverable_check.to_csv(FINAL/"deliverable_file_check.csv", index=False)


,deliverable,status,path
0,Day 2 customer segmentation notebook,Available,notebooks/customer_segmentation.ipynb
1,Day 2 restaurant segmentation notebook,Available,notebooks/restaurant_segmentation.ipynb
2,Day 3 NLP notebook,Not found,notebooks/day3_customer_feedback_nlp.ipynb
3,Day 4 delivery prediction notebook,Not found,notebooks/day4_delivery_time_prediction.ipynb
4,Day 5 demand forecasting notebook,Not found,notebooks/day5_hourly_demand_forecasting.ipynb
5,Day 6 business insights notebook,Not found,notebooks/day6_end_to_end_ml_pipeline_business...
6,Data dictionary,Available,docs/data_dictionary.md
7,Project README,Available,README.md
8,Python requirements,Available,requirements.txt
9,Git ignore rules,Available,.gitignore


## 4. Review model evaluation evidence
**What:** load saved regression and forecasting comparisons if present.

**Result:** actual saved values are displayed; missing files remain marked as unavailable.

**Next:** review segmentation and NLP evidence.

In [4]:
metric_candidates = {
    "Delivery regression comparison": [
        OUTPUTS/"delivery_time_prediction/regression_model_comparison.csv",
        OUTPUTS/"delivery_time_prediction/model_comparison.csv",
        OUTPUTS/"delivery_time_prediction/delivery_model_comparison.csv"],
    "Demand forecasting comparison": [
        OUTPUTS/"demand_forecasting/model_comparison.csv"]
}
metric_status = []
for label, paths in metric_candidates.items():
    found = next((p for p in paths if p.exists()), None)
    if found:
        frame = pd.read_csv(found)
        print("\\n", label, "—", found.relative_to(ROOT))
        display(frame)
        metric_status.append({"component": label, "status": "Available",
                              "source_file": str(found.relative_to(ROOT))})
    else:
        metric_status.append({"component": label, "status": "Missing — run relevant notebook",
                              "source_file": ""})
pd.DataFrame(metric_status).to_csv(FINAL/"model_evidence_check.csv", index=False)


\n Delivery regression comparison — outputs/delivery_time_prediction/regression_model_comparison.csv


,model,MAE,RMSE,R2
0,Ridge,3.938881,4.939502,9.060028e-01
1,Lasso,3.979450,4.974688,9.046589e-01
2,Random Forest,4.646791,5.835968,8.687877e-01
3,Mean baseline,12.627960,16.111119,-1.215141e-07


\n Demand forecasting comparison — outputs/demand_forecasting/model_comparison.csv


,Model,MAE,RMSE,MAPE_percent
0,"ARIMA(1,1,1)",0.751345,0.913677,32.930339
1,Prophet,0.753813,0.917298,37.339180


## 5. Review segmentation and NLP outputs
**What:** check cluster profiles and sentiment outputs saved by previous notebooks.

**Result:** available files are displayed for use in the presentation.

**Next:** check forecast outputs and limitations.

In [5]:
evidence_paths = [
    OUTPUTS/"customer_segmentation/customer_cluster_profiles.csv",
    OUTPUTS/"restaurant_segmentation/restaurant_cluster_profiles.csv",
    OUTPUTS/"nlp_sentiment/sentiment_insights.csv",
    OUTPUTS/"nlp_sentiment/sentiment_model_metrics.csv",
    PROCESSED/"customer_features_clustered.csv",
    PROCESSED/"restaurant_features_clustered.csv"
]
for path in evidence_paths:
    if path.exists():
        print("\\nAvailable:", path.relative_to(ROOT))
        try:
            display(pd.read_csv(path).head(10))
        except Exception as exc:
            print("Could not read:", exc)


\nAvailable: outputs/customer_segmentation/customer_cluster_profiles.csv


,kmeans_cluster,total_orders,average_order_value,total_spending,ordering_frequency_per_month,average_rating_given,weekend_orders,late_night_orders,customer_count,share_percent,business_name
0,0,4.63,498.14,2057.26,2.80,3.61,1.13,1.13,163,54.3,Occasional Budget Customers
1,1,8.07,580.97,4155.70,3.74,3.70,2.34,2.67,128,42.7,Premium Frequent Diners
2,2,1.33,471.42,593.02,33.63,4.26,0.67,0.22,9,3.0,Occasional Budget Customers


\nAvailable: outputs/restaurant_segmentation/restaurant_cluster_profiles.csv


,kmeans_cluster,total_orders,revenue,average_rating,average_prep_time,cancellation_rate,average_delivery_time,restaurant_count,share_percent,business_name
0,0,33.09,14805.19,3.53,28.18,0.08,58.06,11,24.4,Operationally Slower Restaurants
1,1,39.38,19056.50,3.86,15.14,0.08,45.47,21,46.7,Lower-Volume / Developing Restaurants
2,2,46.85,23812.02,3.46,27.92,0.06,59.08,13,28.9,High-Volume Improvement Needed


\nAvailable: outputs/nlp_sentiment/sentiment_insights.csv


...

,model,accuracy,weighted_precision,weighted_recall,weighted_f1
0,TF-IDF + MultinomialNB,1.0,1.0,1.0,1.0


\nAvailable: data/processed/customer_features_clustered.csv


,customer_id,age,gender,city,total_orders,weekend_orders,late_night_orders,total_spending,average_order_value,average_rating_given,ordering_frequency_per_month,kmeans_cluster,dbscan_cluster,pca_1,pca_2
0,CUST0001,22,Male,Pune,8.0,2.0,2.0,3054.03,381.75,3.43,3.03,1,0,0.494955,-0.915690
1,CUST0002,55,Prefer not to say,Pune,6.0,1.0,1.0,3241.89,540.31,3.50,3.08,0,0,-0.227563,-0.104510
2,CUST0003,49,Female,Pune,4.0,3.0,0.0,2328.99,582.25,3.33,3.56,0,0,-0.617954,-0.037336
3,CUST0004,39,Prefer not to say,Hyderabad,1.0,1.0,0.0,750.85,750.85,3.00,30.44,2,-1,-3.131749,2.401382
4,CUST0005,38,Non-binary,Mumbai,2.0,1.0,0.0,1801.33,900.66,3.00,5.54,0,0,-1.452925,0.735679
5,CUST0006,59,Female,Chennai,3.0,0.0,0.0,1528.55,509.52,4.00,7.01,0,0,-2.295664,0.776206
6,CUST0007,22,Male,Bengaluru,3.0,0.0,0.0,811.58,270.53,3.50,1.69,0,0,-2.709727,-1.053294
7,CUST0008,51,Non-binary,Chennai,10.0,1.0,2.0,4940.98,549.00,3.83,4.29,1,0,1.511510,0.305669
8,CUST0009,27,Male,Mumbai,2.0,1.0,0.0,842.42,421.21,4.00,0.92,0,0,-2.382715,-0.075763
9,CUST0010,22,Female,Mumbai,7.0,1.0,4.0,3115.88,445.13,3.75,8.30,1,0,0.448469,-0.130200


\nAvailable: data/processed/restaurant_features_clustered.csv


,restaurant_id,restaurant_name,category,city,restaurant_rating,avg_prep_time_minutes,total_orders,cancelled_orders,average_prep_time,revenue,average_delivery_time,customer_review_average,average_rating,cancellation_rate,kmeans_cluster,dbscan_cluster,pca_1,pca_2
0,REST001,Restaurant 001,Cafe,Mumbai,3.8,18,37,4,18.0,16384.27,46.34,4.000000,4.00,0.108,1,0,-1.663476,-0.549937
1,REST002,Restaurant 002,Chinese,Mumbai,4.0,14,44,6,14.0,20311.45,49.21,3.666667,3.67,0.136,1,0,-0.901545,0.739448
2,REST003,Restaurant 003,Cafe,Chennai,4.1,23,39,1,23.0,20143.21,53.54,3.517241,3.52,0.026,2,0,0.575597,-0.048388
3,REST004,Restaurant 004,Desserts,Bengaluru,3.7,17,46,4,17.0,23259.45,43.62,4.071429,4.07,0.087,1,0,-1.617475,1.597058
4,REST005,Restaurant 005,Healthy,Mumbai,4.3,32,48,1,32.0,25764.34,63.34,3.257143,3.26,0.021,2,0,2.854230,1.432152
5,REST006,Restaurant 006,Cafe,Bengaluru,3.7,33,35,5,33.0,15431.59,60.20,3.277778,3.28,0.143,0,-1,1.730952,-1.600541
6,REST007,Restaurant 007,Chinese,Chennai,3.8,16,43,2,16.0,22986.21,45.13,3.750000,3.75,0.047,1,0,-0.903469,1.145195
7,REST008,Restaurant 008,Desserts,Hyderabad,4.7,22,33,1,22.0,18298.53,50.87,3.913043,3.91,0.030,1,0,-0.655595,-0.780143
8,REST009,Restaurant 009,Pizza,Pune,3.4,35,46,4,35.0,23803.21,66.69,3.400000,3.40,0.087,2,0,2.763909,0.843432
9,REST010,Restaurant 010,Cafe,Pune,2.7,28,34,5,28.0,15276.72,57.89,3.636364,3.64,0.147,0,-1,0.442999,-1.489433


## 6. Check forecasting outputs
**What:** verify whether the next-24-hour forecast and optional Prophet results exist.

**Result:** the notebook reports only files that are actually present.

**Next:** generate a final readiness summary.

In [6]:
forecast_files = [
    OUTPUTS/"demand_forecasting/next_24_hour_demand_forecast.csv",
    OUTPUTS/"demand_forecasting/forecast_comparison.csv",
    OUTPUTS/"demand_forecasting/model_comparison.csv",
    MODELS/"arima_demand_model.joblib",
    MODELS/"prophet_demand_model.joblib"
]
forecast_check = pd.DataFrame([
    {"file": str(p.relative_to(ROOT)), "status": "Available" if p.exists() else "Not found"}
    for p in forecast_files
])
display(forecast_check)
forecast_check.to_csv(FINAL/"forecast_file_check.csv", index=False)


,file,status
0,outputs/demand_forecasting/next_24_hour_demand...,Available
1,outputs/demand_forecasting/forecast_comparison...,Not found
2,outputs/demand_forecasting/model_comparison.csv,Available
3,models/arima_demand_model.joblib,Not found
4,models/prophet_demand_model.joblib,Not found


## 7. Final project readiness summary
**What:** combine file checks into one handover status report.

**Result:** use the report to identify anything to fix before presenting.

**Next:** complete the presentation and Git workflow.

In [7]:
checks = pd.concat([
    raw_status[["file","status"]].rename(columns={"file":"item"}),
    deliverable_check[["deliverable","status"]].rename(columns={"deliverable":"item"})
], ignore_index=True)
checks.to_csv(FINAL/"final_readiness_checklist.csv", index=False)
display(checks)
print("\\nReadiness counts:")
display(checks["status"].value_counts().rename_axis("status").to_frame("items"))
print("\\nFinalization outputs saved to:", FINAL)


,item,status
0,customers.csv,Available
1,restaurants.csv,Available
2,orders.csv,Available
3,deliveries.csv,Available
4,reviews.csv,Available
5,payments.csv,Available
6,drivers.csv,Available
7,Day 2 customer segmentation notebook,Available
8,Day 2 restaurant segmentation notebook,Available
9,Day 3 NLP notebook,Not found


\nReadiness counts:


,items
status,
Available,13
Not found,4


\nFinalization outputs saved to: /home/nineleaps/Downloads/uber_eats-ML/outputs/day7_finalization
